# Step 2 — 貼股價標籤
對每篇文章查找 D+n 個交易日後的收盤價變化，貼上 1（看漲）/ -1（看跌）/ 0（中性）

Big Data & Business Analytics, National Taiwan University

In [1]:
# ── 可調整參數區 ──────────────────────────────────────
COMPANY_ID = '1519'   # 股票代號
N_DAYS     = 3        # 預測幾個交易日後的漲跌
SIGMA      = 0.03     # 漲跌門檻（3% = 0.03），低於此幅度標記為 0
# ──────────────────────────────────────────────────────

import os
from dotenv import load_dotenv
load_dotenv()  # 讀取同目錄下的 .env 檔案

DB_CONFIG = {
    'host':     os.getenv('DB_HOST',    'localhost'),
    'user':     os.getenv('DB_USER',    'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME',    'bda2026'),
    'charset':  os.getenv('DB_CHARSET', 'utf8mb4')
}

In [2]:
import csv
import pymysql
import bisect
from datetime import datetime  # 直接 import 類別

In [3]:
# 從資料庫讀取該公司所有交易日與收盤價
conn = pymysql.connect(**DB_CONFIG)
cursor = conn.cursor()

cursor.execute("""
    SELECT trade_date, closing_price
    FROM stock_prices
    WHERE company_id = %s
    ORDER BY trade_date
""", (COMPANY_ID,))

price_rows = cursor.fetchall()
cursor.close()
conn.close()


trade_dates = [row[0] for row in price_rows]           # 排序好的交易日列表（datetime.date）
price_dict  = {row[0]: float(row[1]) for row in price_rows}  # 日期 → 收盤價
print(f'共 {len(trade_dates)} 個交易日')
print(f'範圍：{trade_dates[0]} ～ {trade_dates[-1]}')

共 483 個交易日
範圍：2023-03-01 ～ 2025-02-27


In [4]:
# [METHOD] 目前方法：固定 n 天後交易日 | 可替換為：事件窗口法（取最大漲跌）

def get_price_on_or_after(date, trade_dates, price_dict):
    """取 date 當天或之後第一個交易日的收盤價"""
    idx = bisect.bisect_left(trade_dates, date)
    if idx >= len(trade_dates):
        return None
    return price_dict[trade_dates[idx]]

def get_nth_trading_day_price(date, n, trade_dates, price_dict):
    """取 date 之後第 n 個交易日（不含當日）的收盤價"""
    idx = bisect.bisect_right(trade_dates, date)  # 第一個 > date 的位置
    target_idx = idx + n - 1                       # 第 n 個交易日
    if target_idx >= len(trade_dates):
        return None
    return price_dict[trade_dates[target_idx]]

In [5]:
# 讀入 articles_raw.csv
articles = []
f = open('articles_raw.csv', newline='', encoding='utf-8')
reader = csv.DictReader(f)
for row in reader:
    articles.append(row)
f.close()

print(f'讀入 {len(articles)} 篇文章')

讀入 3305 篇文章


In [6]:
# 對每篇文章貼標籤
labeled = []
skip_count = 0

for art in articles:
    try:
        post_dt   = datetime.strptime(art['post_time'], '%Y-%m-%d %H:%M:%S')
        post_date = post_dt.date()
    except Exception:
        skip_count += 1
        continue

    price_d  = get_price_on_or_after(post_date, trade_dates, price_dict)   # D 日收盤價
    price_dn = get_nth_trading_day_price(post_date, N_DAYS, trade_dates, price_dict)  # D+n 收盤價

    if price_d is None or price_dn is None or price_d == 0:
        skip_count += 1
        continue

    pct = (price_dn - price_d) / price_d  # 漲跌幅

    if pct > SIGMA:
        label = 1
    elif pct < -SIGMA:
        label = -1
    else:
        label = 0  # 中性，保留但不納入訓練

    art['label']      = label
    art['pct_change'] = round(pct * 100, 2)
    labeled.append(art)

print(f'貼標完成：{len(labeled)} 篇（跳過 {skip_count} 篇）')

貼標完成：3296 篇（跳過 9 篇）


In [7]:
# 儲存 articles_labeled.csv
fieldnames = ['no', 'post_time', 'title', 'content', 's_name', 'label', 'pct_change']

f = open('articles_labeled.csv', 'w', newline='', encoding='utf-8')
writer = csv.DictWriter(f, fieldnames=fieldnames)
writer.writeheader()
for art in labeled:
    writer.writerow({k: art.get(k, '') for k in fieldnames})
f.close()

# 統計
from collections import Counter
label_cnt = Counter(art['label'] for art in labeled)
print(f'看漲 ( 1)：{label_cnt[1]} 篇')
print(f'看跌 (-1)：{label_cnt[-1]} 篇')
print(f'中性 ( 0)：{label_cnt[0]} 篇（不納入訓練）')
print(f'已儲存至 articles_labeled.csv')

看漲 ( 1)：1186 篇
看跌 (-1)：821 篇
中性 ( 0)：1289 篇（不納入訓練）
已儲存至 articles_labeled.csv
